In [1]:
# Kill all processes on the GPU
!fuser -v /dev/nvidia* -k

In [2]:
# Check GPU status
!nvidia-smi

Sat Mar 28 20:45:02 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   60C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
# %%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    %pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    %pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    %pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
%pip install transformers==4.56.2
%pip install --no-deps trl==0.22.2

# Configuration

In [8]:
# Fix TorchDynamo bug
import os
os.environ['TORCHDYNAMO_DISABLE'] = '1'
os.environ['UNSLOTH_DISABLE_COMPILE'] = '1'
os.environ['UNSLOTH_DISABLE_FUSED_KERNELS'] = '1'

In [13]:
# Model configuration
MODEL_ID = 'unsloth/Qwen2.5-VL-7B-Instruct-bnb-4bit'

# Visual codebook configuration
K = 8192

# Model

In [10]:
from unsloth import FastVisionModel
import torch

model, tokenizer = FastVisionModel.from_pretrained(
    model_name = MODEL_ID,
    load_in_4bit = False, # False for LoRA 16bit
    use_gradient_checkpointing = 'unsloth', # True or 'unsloth' for long context
    # max_seq_length = 16384, # Must be this long for VLMs
    # fast_inference = True, # Enable vLLM fast inference
    # fast_inference = False, # Disable to fix the vLLM bug on T4
    # gpu_memory_utilization = 0.8, # Reduce if out of memory
)
print(model)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.17: Fast Qwen2_5_Vl patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Qwen2_5_VLForConditionalGeneration(
  (model): Qwen2_5_VLModel(
    (visual): Qwen2_5_VisionTransformerPretrainedModel(
      (patch_embed): Qwen2_5_VisionPatchEmbed(
        (proj): Conv3d(3, 1280, kernel_size=(2, 14, 14), stride=(2, 14, 14), bias=False)
      )
      (rotary_pos_emb): Qwen2_5_VisionRotaryEmbedding()
      (blocks): ModuleList(
        (0-31): 32 x Qwen2_5_VLVisionBlock(
          (norm1): Qwen2RMSNorm((1280,), eps=1e-06)
          (norm2): Qwen2RMSNorm((1280,), eps=1e-06)
          (attn): Qwen2_5_VLVisionAttention(
            (qkv): Linear(in_features=1280, out_features=3840, bias=True)
            (proj): Linear(in_features=1280, out_features=1280, bias=True)
          )
          (mlp): Qwen2_5_VLMLP(
            (gate_proj): Linear(in_features=1280, out_features=3420, bias=True)
            (up_proj): Linear(in_features=1280, out_features=3420, bias=True)
            (down_proj): Linear(in_features=3420, out_features=1280, bias=True)
            (act_fn): SiLU()

In [11]:
# Copy the original tokenizer for comparison later
import copy

tokenizer_orig = copy.deepcopy(tokenizer)

In [6]:
# print("Vocabulary size:", tokenizer.tokenizer.vocab_size)
# # print(tokenizer.tokenizer.vocab)
# print("Model max length:", tokenizer.tokenizer.model_max_length)
# print("All special tokens (id, token):")
# for z in zip(tokenizer.tokenizer.all_special_ids, tokenizer.tokenizer.all_special_tokens):
#     print(z)
# print("Added vocabulary:")
# for i in tokenizer.tokenizer.get_added_vocab().items():
#     print(i)
# print("Extra special tokens:", tokenizer.tokenizer.extra_special_tokens)

# # 'all_special_ids',
# # 'all_special_tokens',
# # 'all_special_tokens_extended',

In [14]:
vtoks = [f'<|vtok_{i}|>' for i in range(K)]
tokenizer.tokenizer.add_tokens(vtoks);

In [15]:
vtok_start_id, vtok_end_id = tokenizer.tokenizer.encode(f'<|vtok_0|><|vtok_{K-1}|>')
print("Visual token start ID:", vtok_start_id)
print("Visual token end ID:", vtok_end_id)

Visual token start ID: 151665
Visual token end ID: 159856


In [16]:
model.resize_token_embeddings(vtok_end_id+1)
print(model)

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


OutOfMemoryError: CUDA out of memory. Tried to allocate 2.03 GiB. GPU 0 has a total capacity of 14.56 GiB of which 243.81 MiB is free. Including non-PyTorch memory, this process has 14.32 GiB memory in use. Of the allocated memory 14.20 GiB is allocated by PyTorch, and 4.21 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [97]:
print(model.language_model.embed_tokens, ", type:", type(model.language_model.embed_tokens))

Embedding(152064, 3584) , type: <class 'torch.nn.modules.sparse.Embedding'>


In [69]:
model.language_model.embed_tokens.weight[0]

tensor([-0.0171, -0.0013,  0.0189,  ...,  0.0107,  0.0503, -0.0211],
       device='cuda:0', dtype=torch.float16)

In [87]:
model.language_model.embed_tokens.weight[151658].mean()

tensor(5.3644e-07, device='cuda:0', dtype=torch.float16)